# ExaHyPE2 Tutorials, part 1: the acoustic equations

This is the first part of the ExaHyPE2 tutorials. These are defined within the Peano4 documentation, which you can find online at https://hpcsoftware.pages.gitlab.lrz.de/Peano/ or generate locally using `doxygen documentation/Doxyfile`.


This part of the tutorial will briefly go over a few things:

- how to import the Peano4 and ExaHyPE2 API in Python
- the structure of an ExaHyPE2 configuration file
- how to define a PDE solver in ExaHyPE2
- running and visualizing the results of an ExaHyPE2 implementation


This file is accompanied by a "sister file" called **PlanarAcousticSolver.cpp** which contains the C++ implementation of the solver as defined here.


The first thing we will do is importing the Peano4 and ExaHyPE2 API. Python needs to be able to find the Peano4 and ExaHyPE2 python core, so these must be added to the python path, which is the list of locations where python looks for source files.

This can be done in the terminal via `export PYTHONPATH=$PYTHONPATH:$PEANO_ROOT/python`, or directly within the python code as below.
Once on the python path, they can be imported just like any other package:

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("../../../python/"))

import peano4, exahype2

## Defining a solver

Here, we define an ExaHyPE 2 solver, specifically an ADER-DG solver.
It takes as arguments a name, a polynomial order, the unknowns, and the allowable cell sizes for the refinement.

In [ ]:

unknowns = {"p": 1, "v": 2}
auxiliary_variables = {}
cell_h = 1.1 * 2.0 / (3.0 ** 3)

my_solver = exahype2.solvers.aderdg.GlobalAdaptiveTimeStep(
    name                  = "PlanarAcousticSolver",
    order                 = 5,
    min_cell_h            = cell_h,
    max_cell_h            = cell_h,
    time_step_relaxation  = 0.9,
    unknowns              = unknowns,
    auxiliary_variables   = auxiliary_variables
)

my_solver.plot_description = ", ".join(unknowns.keys())

With our solver defined, we can let it know what our equation will look like.
In this case, we will let it know that it only needs a classical flux.
`exahype2.solvers.PDETerms.User_Defined_Implementation` lets the project know that we will manually define the flux in an implementation file.

In [ ]:
my_solver.set_implementation(
    flux = exahype2.solvers.PDETerms.User_Defined_Implementation
)

In addition, let's add some kernel optimizations.
The acoustic equations are linear, which lets us use an optimized version of ADER-DG, so let's specify that.

In [ ]:
my_solver.add_kernel_optimisations(is_linear=True)

## Defining an ExaHyPE2 Project:

Here, we define an ExaHyPE 2 project to which our solver will be bound.
This takes a namespace for the C++ code and a name for the executable.
We then add our solver to this project and, just for cleanliness, set an output path for our results.

In [ ]:
exahype2_project = exahype2.Project(
    namespace     = ["exahype2", "training", "acoustic"],
    directory     = ".",
    project_name  = "PlanarWaves",
    executable    = "PlanarWaves"
)

exahype2_project.add_solver(my_solver)
exahype2_project.set_output_path("solutions")

Now that we have an ExaHyPE 2 project, let's populate it with some information:
This is essentially the domain our solver will run in.
The project needs to know the domain size, how long the simulation should run,
how often it should plot the solution, and what type of boundary we use.

In [ ]:
exahype2_project.set_global_simulation_parameters(
    dimensions            = 2,
    size                  = [2.0,  2.0],
    offset                = [-1.0, -1.0],
    min_end_time          = 1.414,  # sqrt(2)
    max_end_time          = 1.414,  # sqrt(2)
    first_plot_time_stamp = 0.0,
    time_in_between_plots = 0.1,
    periodic_BC           = [True, True],
)

Finally, we specify some optimizations for parallel runs and compile the project.

In [ ]:
exahype2_project.set_load_balancing(
    "toolbox::loadbalancing::strategies::SpreadOutOnceGridStagnates",
    "new ::exahype2::LoadBalancingConfiguration()"
)

exahype2_project.set_Peano4_installation("../../../", peano4.output.CompileMode.Release)
peano4_project = exahype2_project.generate_Peano4_project()
peano4_project.generate()
peano4_project.build(make=True, make_clean_first=False, throw_away_data_after_build=False)

Once the implementation has compiled, we can run it.

This is usually done directly in the terminal via `./$executable` but can be done in python as well.

In [ ]:
import subprocess

subprocess.run("./PlanarWaves", shell = True, executable="/bin/bash")

## Visualisation

Finally, once the program has finished running, we can convert the output from the native peano4 format into something that can be visualized.

Peano4 provides a script for this, which can be called via `python3 $PEANO_ROOT/python/peano4/visualisation/render.py` followed by the name of the files we want to convert.

This relies on the **paraview** python libraries, and converts the files into .vtu files, which can be visualised for example using paraview.


In [ ]:
import subprocess
subprocess.call(
    ["python3", "../../../python/peano4/visualisation/render.py", "solutions/solution-PlanarAcousticSolver.peano-patch-file"]
)